Example of Jupyter "magic command":

In [37]:
%pwd

'/Users/jb160-local/GitHub/gitRDS/EDS213/bren-eds213/modules/week06'

To install DuckDB Python module:

In [38]:
# %pip install duckdb
# %pip install pandas

In [7]:

import duckdb
import pandas as pd

### 1. Create a connection to the database

In [17]:
conn = duckdb.connect("~/GitHub/gitRDS/EDS213/bren-eds213-data/database/database.duckdb")

In [18]:
conn

### 2. Now let's query the databse using:
  a. `.sql()` method -- specific to DuckDB and analysis focused


In [29]:
conn.sql("SELECT * FROM Site LIMIT 5")


┌─────────┬─────────────────────┬─────────────────┬──────────┬───────────┬───────┐
│  Code   │      Site_name      │    Location     │ Latitude │ Longitude │ Area  │
│ varchar │       varchar       │     varchar     │  float   │   float   │ float │
├─────────┼─────────────────────┼─────────────────┼──────────┼───────────┼───────┤
│ barr    │ Barrow              │ Alaska, USA     │     71.3 │    -156.6 │ 220.4 │
│ burn    │ Burntpoint Creek    │ Ontario, Canada │     55.2 │     -84.3 │  63.0 │
│ bylo    │ Bylot Island        │ Nunavut, Canada │     73.2 │     -80.0 │ 723.6 │
│ cakr    │ Cape Krusenstern    │ Alaska, USA     │     67.1 │    -163.5 │  54.1 │
│ cari    │ Canning River Delta │ Alaska, USA     │     70.1 │    -145.8 │ 722.0 │
└─────────┴─────────────────────┴─────────────────┴──────────┴───────────┴───────┘

Note: This is also a lazy evaluation like we were doing with `dbplyr`. At this point, the data has not been fully processed or brought into Python memory yet. It's a symbolic representation of a query. To see the actual data, you need to materialize the relation. You can do this in several ways: 

  > `.show()`: Prints a nice tabular representation (great for interactive exploration).
  > `.fetchall()`: Returns all results as a list of tuples (the traditional DB-API way).
  > `.fetchone()`: Returns the next single row as a tuple.
  >`.df()`: Converts the result into a Pandas DataFrame.Now we want results... 


In [30]:
# You get a list of tuples (one per row)
conn.sql("SELECT * FROM Site LIMIT 5").fetchall()


[('barr',
  'Barrow',
  'Alaska, USA',
  71.30000305175781,
  -156.60000610351562,
  220.39999389648438),
 ('burn',
  'Burntpoint Creek',
  'Ontario, Canada',
  55.20000076293945,
  -84.30000305175781,
  63.0),
 ('bylo',
  'Bylot Island',
  'Nunavut, Canada',
  73.19999694824219,
  -80.0,
  723.5999755859375),
 ('cakr',
  'Cape Krusenstern',
  'Alaska, USA',
  67.0999984741211,
  -163.5,
  54.099998474121094),
 ('cari',
  'Canning River Delta',
  'Alaska, USA',
  70.0999984741211,
  -145.8000030517578,
  722.0)]

In [40]:
# You get a pandas dataframe
site_df = conn.sql("SELECT * FROM Site").df()
site_df.head()

,Code,Site_name,Location,Latitude,Longitude,Area
0,barr,Barrow,"Alaska, USA",71.300003,-156.600006,220.399994
1,burn,Burntpoint Creek,"Ontario, Canada",55.200001,-84.300003,63.000000
2,bylo,Bylot Island,"Nunavut, Canada",73.199997,-80.000000,723.599976
3,cakr,Cape Krusenstern,"Alaska, USA",67.099998,-163.500000,54.099998
4,cari,Canning River Delta,"Alaska, USA",70.099998,-145.800003,722.000000


In [32]:
site_table = conn.execute("SELECT * FROM Site")

site_table

  b. `.execute()` method -- more ubiquitous to other database workflows where you create a cursor object that you use to iterate on the rows of a table


In [34]:
cur = conn.cursor()

Cursors don't store anything, they just transfer queries to the database and get results back.

In [35]:
cur.execute("SELECT * FROM Site LIMIT 5")

We still need to fectch the results as before:

In [36]:
cur.fetchall()

[('barr',
  'Barrow',
  'Alaska, USA',
  71.30000305175781,
  -156.60000610351562,
  220.39999389648438),
 ('burn',
  'Burntpoint Creek',
  'Ontario, Canada',
  55.20000076293945,
  -84.30000305175781,
  63.0),
 ('bylo',
  'Bylot Island',
  'Nunavut, Canada',
  73.19999694824219,
  -80.0,
  723.5999755859375),
 ('cakr',
  'Cape Krusenstern',
  'Alaska, USA',
  67.0999984741211,
  -163.5,
  54.099998474121094),
 ('cari',
  'Canning River Delta',
  'Alaska, USA',
  70.0999984741211,
  -145.8000030517578,
  722.0)]

Always get tuples, even if you only request one column

In [56]:
cur.execute("SELECT Nest_ID FROM Bird_nests LIMIT 10")

In [57]:
cur.fetchall()

[('14HPE1',),
 ('11eaba',),
 ('11eabaagc01',),
 ('11eabaagv01',),
 ('11eababbc02',),
 ('11eababsv01',),
 ('11eabaduh01',),
 ('11eabaduv01',),
 ('11eabarpc01',),
 ('11eabarpc02',)]

In [58]:
cur.execute("SELECT Nest_ID FROM Bird_nests LIMIT 10")
[t[0] for t in cur.fetchall()]

['14HPE1',
 '11eaba',
 '11eabaagc01',
 '11eabaagv01',
 '11eababbc02',
 '11eababsv01',
 '11eabaduh01',
 '11eabaduv01',
 '11eabarpc01',
 '11eabarpc02']

3. Get the one result

In [ ]:
cur.execute("SELECT COUNT(*) FROM Bird_nests")
cur.fetchall()

[(1547,)]

In [ ]:
cur.execute("SELECT COUNT(*) FROM Bird_nests")
cur.fetchone()

(1547,)

In [ ]:
cur.execute("SELECT COUNT(*) FROM Bird_nests")
cur.fetchone()[0]

1547

3. Using an iterator - but DuckDB doesn't support iterators :(

In [ ]:
cur.execute("SELECT Nest_ID FROM Bird_nests LIMIT 10")
for row in cur:
    print(f"got {row[0]}")

TypeError: '_duckdb.DuckDBPyConnection' object is not iterable

A workaround is to use the `.fectchone()` method:

In [41]:
cur.execute("SELECT Nest_ID FROM Bird_nests LIMIT 10")
while True:
    row = cur.fetchone()
    if row == None:
        break
    # do something with row
    print(f"got nest ID {row[0]}")

got nest ID 14HPE1
got nest ID 11eaba
got nest ID 11eabaagc01
got nest ID 11eabaagv01
got nest ID 11eababbc02
got nest ID 11eababsv01
got nest ID 11eabaduh01
got nest ID 11eabaduv01
got nest ID 11eabarpc01
got nest ID 11eabarpc02


Can do things other than SELECT!

In [59]:
cur.execute("CREATE TEMP TABLE temp_table AS
            SELECT * FROM Bird_nests LIMIT 10")

SyntaxError: unterminated string literal (detected at line 1) (1747419494.py, line 1)

**Watch out for line breaks!!**

In [61]:
cur.execute("""
    CREATE TEMP TABLE temp_table AS
    SELECT * FROM Bird_nests LIMIT 10
    """)

CatalogException: Catalog Error: Table with name "temp_table" already exists!

In [67]:
cur.execute("SELECT * FROM temp_table")

In [68]:
cur.fetchone()

('b14.6',
 2014,
 'chur',
 '14HPE1',
 'sepl',
 'vloverti',
 datetime.date(2014, 6, 14),
 None,
 3,
 None,
 None)

A note on fragility

For example:
INSERT INTO Site VALUES ("abcd", "Foo", 35.7, 42.3, "?")

A less fragile way of expressing the same thing:
INSERT INTO Site (Code, Site_name, Latitude, Longitude, Something_else)
   VALUES ("abcd", "Foo", 35.7, 42.3, "?")
   
In the same vein: SELECT * is fragile

In [ ]:
cur.execute("SELECT * FROM Site LIMIT 3")
cur.fetchall()

[('barr',
  'Barrow',
  'Alaska, USA',
  71.30000305175781,
  -156.60000610351562,
  220.39999389648438),
 ('burn',
  'Burntpoint Creek',
  'Ontario, Canada',
  55.20000076293945,
  -84.30000305175781,
  63.0),
 ('bylo',
  'Bylot Island',
  'Nunavut, Canada',
  73.19999694824219,
  -80.0,
  723.5999755859375)]

A better, more robust way of coding the same thing:

In [ ]:
cur.execute("SELECT Site_name, Code, Latitude, Longitude FROM Site LIMIT 3")
cur.fetchall()

[('Barrow', 'barr', 71.30000305175781, -156.60000610351562),
 ('Burntpoint Creek', 'burn', 55.20000076293945, -84.30000305175781),
 ('Bylot Island', 'bylo', 73.19999694824219, -80.0)]

An extended example: Question we're trying to answer: How many nests do we have for each species?

Approach: first get all species.  Then execute a count query for each species.

A digression: string interpolation in Python

In [ ]:
# The % method
s = "My name is %s"
print(s % "Greg")
s = "My name is %s and the other teacher's name is %s"
print(s % ("Greg", "Julien"))
# The new f-string method
name = "Greg"
print(f"My name is {name}")
# Third way
print("My name is {}".format("Greg"))

My name is Greg
My name is Greg and the other teacher's name is Julien
My name is Greg
My name is Greg


In [ ]:
query = """
   SELECT COUNT(*) FROM Bird_nests
   WHERE Species = '%s'
"""
cur.execute("SELECT Code FROM Species LIMIT 3")
for row in cur.fetchall():  # DuckDB workaround
    code = row[0]
    prepared_query = query % code
    #print(prepared_query)
    cur2 = conn.cursor()
    cur2.execute(prepared_query)
    print(f"Species {code} has {cur2.fetchone()[0]} nests")
    cur2.close()

Species agsq has 0 nests
Species amcr has 0 nests
Species amgp has 29 nests


The above Python interpolation is dangerous and has caused many database hacks!  There's a better way

In [ ]:
query = """
   SELECT COUNT(*) FROM Bird_nests
   WHERE Species = ?
"""
cur.execute("SELECT Code FROM Species LIMIT 3")
for row in cur.fetchall():  # DuckDB workaround
    code = row[0]
    # NOT NEEDED! prepared_query = query % code
    #print(prepared_query)
    cur2 = conn.cursor()
    cur2.execute(query, [code])  # <-- added argument here
    print(f"Species {code} has {cur2.fetchone()[0]} nests")
    cur2.close()

Species agsq has 0 nests
Species amcr has 0 nests
Species amgp has 29 nests


Let's illustrate the danger with a different example

In [ ]:
abbrev = "TS"
name = "Taylor Swift"
cur.execute("""
   INSERT INTO Personnel (Abbreviation, Name)
   VALUES ('%s', '%s')
   """ % (abbrev, name)
           )

In [ ]:
cur.execute("SELECT * FROM Personnel")
cur.fetchall()[-3:]

[('emagnuson', 'Emily Magnuson'),
 ('mcorrell', 'Maureen Correll'),
 ('TS', 'Taylor Swift')]

In [ ]:
abbrev = "CO"
name = "Conan O'Brien"
cur.execute("""
   INSERT INTO Personnel (Abbreviation, Name)
   VALUES ('%s', '%s')
   """ % (abbrev, name)
           )

ParserException: Parser Error: syntax error at or near "Brien"

LINE 3:    VALUES ('CO', 'Conan O'Brien')
                                  ^

In [ ]:
"""
   INSERT INTO Personnel (Abbreviation, Name)
   VALUES ('%s', '%s')
   """ % (abbrev, name)

"\n   INSERT INTO Personnel (Abbreviation, Name)\n   VALUES ('CO', 'Conan O'Brien')\n   "

In [ ]:
abbrev = "CO"
name = "Conan O'Brien"
cur.execute("""
   INSERT INTO Personnel (Abbreviation, Name)
   VALUES (?, ?)
   """,
    [abbrev, name])

In [ ]:
cur.execute("SELECT * FROM Personnel")
cur.fetchall()[-3:]

[('mcorrell', 'Maureen Correll'),
 ('TS', 'Taylor Swift'),
 ('CO', "Conan O'Brien")]

Don't forget to close your connection when you are done!

In [ ]:
conn.close()